####Paso 1 - Leer los archivos CSV usando "DataFrameReader de Spark"

In [0]:
%run "../includes/configurations"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
movies_companies_schema = StructType(fields=[
  StructField("movieId", IntegerType(), True),
  StructField("companyId", StringType(), True)
])

In [0]:
movies_companies_df = spark.read \
  .schema(movies_companies_schema) \
  .csv(f"{bronze_folder_path}/{v_file_date}/movie_company")

In [0]:
display(movies_companies_df)

####Paso 2 - Renombrar las columnas y añadir nuevas columnas

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
movies_companies_final_df = movies_companies_df \
    .withColumnsRenamed({"movieId": "movie_id",
                        "companyId": "company_id"}) \
    .withColumn("ingestion_date", current_timestamp()) \
    .withColumn("enviroment", lit("Produccion")) \
    .withColumn("file_date", lit(v_file_date))

In [0]:
display(movies_companies_final_df)

####Paso 3 - Escribir la salida en formato "Parquet"

In [0]:
#overwrite_partition("movie_silver", "movie_companies", "file_date", v_file_date)

In [0]:
#movies_companies_final_df.write.mode("overwrite").parquet("abfss://silver@moviehistorybymg.dfs.core.windows.net/movie_company")
#movies_companies_final_df.write.mode("append").partitionBy("file_date").format("delta").saveAsTable("movie_silver.movie_companies")
merge_condition = 'tgt.movie_id = src.movie_id AND tgt.company_id = src.company_id AND tgt.file_date = src.file_date'
merge_delta_lake (movies_companies_final_df, "movie_silver", "movie_companies", merge_condition, "file_date")

In [0]:
%sql
SELECT file_date, COUNT(1)
FROM movie_silver.movie_companies
GROUP BY file_date;

In [0]:
%sql
SELECT *
FROM movie_silver.movie_companies;

In [0]:
#%fs
#ls abfss://silver@moviehistorybymg.dfs.core.windows.net/movie_company

In [0]:
#display(spark.read.parquet("abfss://silver@moviehistorybymg.dfs.core.windows.net/movie_company"))